# ML Testing & MLOps: Building a Promotion Gate for a Deployed Classifier

## Problem statement

You are the QA engineer on a team that just trained a new candidate model to predict Titanic
passenger survival, and it is about to replace the model currently running in production. Your
job is not to train a better model — it's to decide, automatically, whether this candidate is
*safe to deploy*. In a real MLOps pipeline this decision is made by a **promotion gate**: a suite
of automated checks that runs in CI on every new model artifact before it is allowed into
production, exactly the way a test suite gates a code deploy.

**Dataset:** the real [Titanic](https://www.kaggle.com/c/titanic) passenger manifest (891 real
passenger records: class, sex, age, fare, family size, port of embarkation, survival outcome),
loaded from the public `mwaskom/seaborn-data` mirror on GitHub. It has real missing data and
real class imbalance, which is exactly what makes naive QA checks on it misleading — you'll see
this directly in Part 2.

### What you should produce

A `run_promotion_gate(...)` function, built from three independent checks, each catching a
different class of bug that a single "is accuracy OK?" check would miss:

1. **`validate_batch(df, schema)`** — a data-contract validator for *incoming data*, not the
   model. It checks required columns are present, categorical columns only contain allowed
   values, numeric columns fall within sane ranges, and missing-value rates stay under
   calibrated thresholds. It must return a list of concrete violation messages (not just
   True/False) and must correctly flag a deliberately corrupted copy of the batch while passing
   the clean one.
2. **`evaluate_with_slices(model, X_test, y_test, slice_cols, baseline_metrics, tol_acc, tol_f1)`**
   — a *performance regression* gate that compares the candidate model's accuracy **and F1**
   on each demographic slice (by `sex`, by `pclass`) against a stored baseline from the
   last approved model, flagging any slice that regressed by more than a tolerance. You must
   demonstrate a case where overall accuracy barely moves but one slice's F1 collapses — and
   explain why comparing against accuracy alone would have missed it.
3. **`check_determinism(model_factory, X_train, y_train, X_eval, n_runs)`** — refits the same
   model `n_runs` times and asserts predictions and predicted probabilities are bit-for-bit
   identical across runs, catching a model that was accidentally left unseeded.

Finally, wire all three into `run_promotion_gate(...)`, which runs every check and returns a
single pass/fail report with every failing reason listed — the kind of output that would block
a CI pipeline and tell the on-call engineer exactly what broke.


## Setup

Imports and data loading. Run this cell as-is.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
raw = pd.read_csv(DATA_URL)

FEATURE_COLS = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
df = raw[["survived"] + FEATURE_COLS].copy()

print(df.shape)
df.head()

## Part 1 — Data contract validator

Fill in `validate_batch`. Calibrate `SCHEMA`'s missing-value thresholds from `df`'s own
missing rates (print `df.isna().mean()` first) rather than guessing — a contract test should be
grounded in what "normal" actually looks like for this pipeline, with a little headroom.

In [ ]:
# TODO: inspect real missing rates and value ranges to calibrate the schema below
print(df.isna().mean())
print(df["pclass"].unique(), df["sex"].unique(), df["embarked"].unique())
print(df["age"].describe())
print(df["fare"].describe())

In [ ]:
SCHEMA = {
    "required_columns": ["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"],
    "categorical_allowed": {
        # TODO: fill in the allowed category sets for sex, pclass, embarked, survived
    },
    "numeric_ranges": {
        # TODO: fill in sane (lo, hi) ranges for age, fare, sibsp, parch
    },
    "max_missing_rate": {
        # TODO: fill in a calibrated max missing-value rate per column
    },
}


def validate_batch(batch: pd.DataFrame, schema: dict) -> list[str]:
    '''Return a list of human-readable contract violations found in `batch` (empty if clean).'''
    violations = []
    # TODO: 1) check required columns are present
    # TODO: 2) for present categorical columns, flag any values outside the allowed set
    # TODO: 3) for present numeric columns, flag any values outside the allowed range
    # TODO: 4) for present columns, flag missing-value rate above the calibrated threshold
    return violations


# Sanity checks you should be able to run once validate_batch is implemented:
# violations_clean = validate_batch(df, SCHEMA)
# assert violations_clean == [], violations_clean
#
# corrupted = df.copy()
# corrupted.loc[0:50, "age"] = -5
# corrupted.loc[51, "sex"] = "unknown"
# corrupted.loc[52:400, "embarked"] = np.nan
# violations_corrupted = validate_batch(corrupted, SCHEMA)
# assert len(violations_corrupted) >= 3, violations_corrupted
# violations_corrupted

## Part 2 — Slice-based performance regression gate

First build a train/test split and a baseline model (provided). Then fill in `slice_metrics`
and `evaluate_with_slices`, which compare a *candidate* model's per-slice accuracy **and F1**
against the baseline's per-slice metrics — not against the candidate's own overall metric.

In [ ]:
X = df.drop(columns=["survived"])
y = df["survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

NUMERIC_COLS = ["age", "sibsp", "parch", "fare"]
CATEGORICAL_COLS = ["pclass", "sex", "embarked"]


def make_preprocessor():
    # Built fresh on every call -- a ColumnTransformer/Pipeline step is stateful once fit,
    # so reusing one instance across two Pipelines would let fitting the second silently
    # overwrite the fitted statistics (medians, OHE categories) the first one relies on.
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), NUMERIC_COLS),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL_COLS),
    ])


def make_model(random_state=42):
    return Pipeline([
        ("preprocess", make_preprocessor()),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=random_state)),
    ])


baseline_model = make_model()
baseline_model.fit(X_train, y_train)

In [ ]:
def slice_metrics(model, X_eval, y_eval, slice_cols, min_slice_size=15) -> dict:
    '''Return {"overall": {...}, "<col>=<value>": {...}, ...} with accuracy/f1/n per slice.'''
    # TODO: predict on X_eval, compute overall accuracy + F1
    # TODO: for each column in slice_cols, for each value with at least min_slice_size rows,
    #       compute accuracy + F1 restricted to that slice
    raise NotImplementedError


def evaluate_with_slices(model, X_eval, y_eval, slice_cols, baseline_metrics,
                          tol_acc=0.08, tol_f1=0.10, min_slice_size=15) -> dict:
    '''Compare model's slice_metrics against baseline_metrics; return current metrics + failures.'''
    # TODO: compute current = slice_metrics(...)
    # TODO: for every key baseline_metrics shares with current, flag a failure if
    #       baseline_accuracy - current_accuracy > tol_acc, OR baseline_f1 - current_f1 > tol_f1
    raise NotImplementedError


# baseline_metrics = slice_metrics(baseline_model, X_test, y_test, ["sex", "pclass"])
# baseline_metrics

Now simulate a realistic staging bug: an upstream data pipeline change accidentally drops
almost all third-class passengers before training. Check what `evaluate_with_slices` reports
for this candidate against the baseline — and compare that to what a check based only on
overall accuracy would have told you.

In [ ]:
# A pipeline bug: keep only ~3% of pclass == 3 training rows
keep_mask = ~(
    (X_train["pclass"] == 3) & (np.random.RandomState(1).rand(len(X_train)) < 0.97)
)
X_train_buggy = X_train[keep_mask]
y_train_buggy = y_train[keep_mask]
print("pclass==3 rows kept:", (X_train_buggy["pclass"] == 3).sum(), "of", (X_train["pclass"] == 3).sum())

candidate_model = make_model()
candidate_model.fit(X_train_buggy, y_train_buggy)

# TODO: call evaluate_with_slices(candidate_model, X_test, y_test, ["sex", "pclass"], baseline_metrics)
# TODO: print the overall accuracy of baseline_model vs candidate_model on X_test, y_test
# TODO: in a markdown note below (or a comment), explain what the overall-accuracy-only
#       comparison would have shown vs what the slice gate actually caught

## Part 3 — Determinism check

A model that silently depends on unseeded randomness is a production risk: the same input can
get a different prediction on redeploy. Fill in `check_determinism`.

In [ ]:
def check_determinism(model_factory, X_fit, y_fit, X_eval, n_runs=3, random_state=42, atol=1e-9):
    '''Fit model_factory(random_state=random_state) n_runs times on (X_fit, y_fit).
    Return (True, msg) if predict() and predict_proba() on X_eval are identical/close across all
    runs, else (False, msg) naming the first run that diverged.'''
    # TODO: implement
    raise NotImplementedError


def make_model_unseeded(random_state=None):
    # A buggy variant: ignores the random_state argument entirely.
    return Pipeline([
        ("preprocess", preprocess),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=None)),
    ])


# TODO: run check_determinism(make_model, X_train, y_train, X_test) -> should be deterministic
# TODO: run check_determinism(make_model_unseeded, X_train, y_train, X_test) -> should NOT be

## Part 4 — Wire it all together

Build `run_promotion_gate(...)` that runs all three checks against a candidate model + an
incoming data batch, and returns/prints a single PASS/FAIL report naming every failure.

In [ ]:
def run_promotion_gate(candidate_model, incoming_batch, schema, X_eval, y_eval,
                        slice_cols, baseline_metrics, model_factory,
                        X_fit, y_fit) -> dict:
    '''Return {"passed": bool, "failures": [...], "details": {...}}.'''
    # TODO: run validate_batch on incoming_batch
    # TODO: run evaluate_with_slices on candidate_model
    # TODO: run check_determinism on model_factory
    # TODO: aggregate into one report; print a clear PASS/FAIL summary with every reason
    raise NotImplementedError


# TODO: run run_promotion_gate once for the clean baseline_model (should PASS)
# TODO: run run_promotion_gate once for the buggy candidate_model (should FAIL, naming the slice)

---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

### Part 1 — data contract validator

```python
SCHEMA = {
    "required_columns": ["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"],
    "categorical_allowed": {
        "sex": {"male", "female"},
        "pclass": {1, 2, 3},
        "embarked": {"S", "C", "Q"},
        "survived": {0, 1},
    },
    "numeric_ranges": {
        "age": (0, 100),
        "fare": (0, 600),
        "sibsp": (0, 10),
        "parch": (0, 10),
    },
    "max_missing_rate": {
        # age is genuinely ~20% missing in this data, so give it headroom; everything else
        # is normally complete or near-complete, so a small cushion catches a real regression.
        "age": 0.25,
        "embarked": 0.05,
        "fare": 0.01,
        "sex": 0.0, "pclass": 0.0, "survived": 0.0, "sibsp": 0.0, "parch": 0.0,
    },
}


def validate_batch(batch, schema):
    violations = []
    for col in schema["required_columns"]:
        if col not in batch.columns:
            violations.append(f"missing required column: {col}")
    present = [c for c in schema["required_columns"] if c in batch.columns]

    for col, allowed in schema["categorical_allowed"].items():
        if col not in present:
            continue
        bad = set(batch[col].dropna().unique()) - allowed
        if bad:
            violations.append(f"{col}: unexpected categories {bad}")

    for col, (lo, hi) in schema["numeric_ranges"].items():
        if col not in present:
            continue
        s = batch[col].dropna()
        out = s[(s < lo) | (s > hi)]
        if len(out) > 0:
            violations.append(f"{col}: {len(out)} values out of range [{lo}, {hi}]")

    for col, max_rate in schema["max_missing_rate"].items():
        if col not in present:
            continue
        rate = batch[col].isna().mean()
        if rate > max_rate:
            violations.append(f"{col}: missing rate {rate:.1%} exceeds allowed {max_rate:.1%}")

    return violations
```

The clean `df` returns `[]`; a copy with negative ages, an invalid `sex` value, and 40% of
`embarked` nulled returns three concrete violations — one per injected fault. The point of
injecting synthetic corruption here isn't to model anything real, it's to prove the validator
actually fires on each fault class rather than only ever passing.

### Part 2 — slice regression gate

```python
def slice_metrics(model, X_eval, y_eval, slice_cols, min_slice_size=15):
    preds = model.predict(X_eval)
    out = {"overall": {"accuracy": accuracy_score(y_eval, preds),
                        "f1": f1_score(y_eval, preds), "n": len(y_eval)}}
    for col in slice_cols:
        for value in sorted(X_eval[col].dropna().unique(), key=str):
            mask = X_eval[col] == value
            n = int(mask.sum())
            if n < min_slice_size:
                continue
            out[f"{col}={value}"] = {
                "accuracy": accuracy_score(y_eval[mask], preds[mask]),
                "f1": f1_score(y_eval[mask], preds[mask], zero_division=0),
                "n": n,
            }
    return out


def evaluate_with_slices(model, X_eval, y_eval, slice_cols, baseline_metrics,
                          tol_acc=0.08, tol_f1=0.10, min_slice_size=15):
    current = slice_metrics(model, X_eval, y_eval, slice_cols, min_slice_size)
    failures = []
    for key, base in baseline_metrics.items():
        cur = current.get(key)
        if cur is None:
            continue
        if base["accuracy"] - cur["accuracy"] > tol_acc:
            failures.append(f"{key}: accuracy {base['accuracy']:.3f} -> {cur['accuracy']:.3f}")
        if base["f1"] - cur["f1"] > tol_f1:
            failures.append(f"{key}: F1 {base['f1']:.3f} -> {cur['f1']:.3f}")
    return {"current": current, "failures": failures}


baseline_metrics = slice_metrics(baseline_model, X_test, y_test, ["sex", "pclass"])
result = evaluate_with_slices(candidate_model, X_test, y_test, ["sex", "pclass"], baseline_metrics)
result["failures"]
# -> ['pclass=3: F1 0.426 -> 0.205']
```

Dropping 97% of third-class training rows barely moves **overall** accuracy (0.749 → 0.758 —
it even looks slightly *better*, because third class is the majority slice and the model now
just leans harder on the majority "did not survive" outcome). An accuracy-only gate comparing
the candidate's own overall number to a fixed bar would have passed this model straight into
production. The slice gate catches it because it compares *per-slice* F1 against the *same
slice's* own baseline — third class F1 collapses from 0.43 to 0.21, a huge relative drop,
while every other slice stays within tolerance. This is also why the gate compares against a
stored baseline rather than against the candidate's own overall metric in the same run: some
slices (like `sex=male` here) naturally have much lower F1 than the dataset overall simply
because of how imbalanced survival is within that slice — that gap is real but not a
regression, and a baseline comparison is what tells the two apart.

### Part 3 — determinism check

```python
def check_determinism(model_factory, X_fit, y_fit, X_eval, n_runs=3, random_state=42, atol=1e-9):
    first_preds, first_proba = None, None
    for i in range(n_runs):
        m = model_factory(random_state=random_state)
        m.fit(X_fit, y_fit)
        preds, proba = m.predict(X_eval), m.predict_proba(X_eval)
        if first_preds is None:
            first_preds, first_proba = preds, proba
            continue
        if not np.array_equal(preds, first_preds):
            return False, f"predictions differ on run {i + 1} vs run 1"
        if not np.allclose(proba, first_proba, atol=atol):
            return False, f"predicted probabilities differ on run {i + 1} vs run 1"
    return True, "deterministic across all runs"
```

`check_determinism(make_model, ...)` returns `(True, "deterministic across all runs")` because
`random_state` is threaded through to the `RandomForestClassifier`. Swap in
`make_model_unseeded` (which ignores `random_state` and leaves the forest unseeded) and it
returns `(False, "predictions differ on run 2 vs run 1")` — same data, same code path,
different answer on every refit.

### Part 4 — the gate

```python
def run_promotion_gate(candidate_model, incoming_batch, schema, X_eval, y_eval,
                        slice_cols, baseline_metrics, model_factory, X_fit, y_fit):
    failures = []

    batch_violations = validate_batch(incoming_batch, schema)
    failures += [f"[data contract] {v}" for v in batch_violations]

    slice_result = evaluate_with_slices(candidate_model, X_eval, y_eval, slice_cols, baseline_metrics)
    failures += [f"[slice regression] {f}" for f in slice_result["failures"]]

    det_ok, det_msg = check_determinism(model_factory, X_fit, y_fit, X_eval)
    if not det_ok:
        failures.append(f"[determinism] {det_msg}")

    passed = len(failures) == 0
    report = {"passed": passed, "failures": failures, "slice_metrics": slice_result["current"]}
    print("PROMOTION GATE:", "PASS" if passed else "FAIL")
    for f in failures:
        print(" -", f)
    return report


# Clean candidate: PASS
run_promotion_gate(baseline_model, df, SCHEMA, X_test, y_test, ["sex", "pclass"],
                    baseline_metrics, make_model, X_train, y_train)

# Buggy candidate: FAIL, naming the exact slice that regressed
run_promotion_gate(candidate_model, df, SCHEMA, X_test, y_test, ["sex", "pclass"],
                    baseline_metrics, make_model, X_train_buggy, y_train_buggy)
```

The second call fails with a single, specific reason —
`[slice regression] pclass=3: F1 0.426 -> 0.205` — which is exactly the kind of actionable
signal a promotion gate should produce: not "something's wrong" but "this slice, this metric,
this magnitude."

</details>
